# Unsupervised Learning - Clustering

This notebook applies K-Means and DBSCAN clustering on the Titanic dataset, uses the elbow method for choosing the number of clusters, visualizes clusters with PCA, and summarizes the insights for Zynxis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.datasets import load_titanic
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

# Load dataset
try:
    df = pd.read_csv('titanic.csv')
except FileNotFoundError:
    data = load_titanic(as_frame=True)
    df = data.frame

print(df.head())
print(df.shape)

In [ ]:
# Prepare features
df_model = df.copy()
df_model = df_model.drop(columns=['survived'], errors='ignore')

categorical_cols = df_model.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = df_model.select_dtypes(include=['number']).columns.tolist()

if categorical_cols:
    df_model[categorical_cols] = df_model[categorical_cols].fillna('missing')
    df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

imputer = SimpleImputer(strategy='median')
scaled_values = imputer.fit_transform(df_model[numeric_cols])
scaler = StandardScaler()
scaled_values = scaler.fit_transform(scaled_values)

X = pd.DataFrame(scaled_values, columns=numeric_cols, index=df_model.index)
X.head()

In [ ]:
# Elbow method
inertias = []
for k in range(1, 9):
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    model.fit(X)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 9), inertias, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# K-Means
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
kmeans_labels = kmeans.fit_predict(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='viridis', alpha=0.7)
plt.title('K-Means Clusters (PCA)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# DBSCAN
dbscan = DBSCAN(eps=2.5, min_samples=10)
dbscan_labels = dbscan.fit_predict(X)

plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=dbscan_labels, cmap='viridis', alpha=0.7)
plt.title('DBSCAN Clusters (PCA)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('dbscan_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Export results
result_df = df.copy()
result_df['kmeans_cluster'] = kmeans_labels
result_df['dbscan_cluster'] = dbscan_labels
result_df.to_csv('clustered_dataset.csv', index=False)

summary = result_df.groupby('kmeans_cluster').mean(numeric_only=True).reset_index()
summary.to_csv('cluster_summary.csv', index=False)
summary

## Business Insight for Zynxis

These clusters can help Zynxis group interns by similar performance patterns, segment users based on behavior, or identify unusual groups that may need special attention.